# Model comparison

Report metrics from SQLite. Table 1 includes saved games on SQLite. Table 2 covers eligible games on trace store.

In [ ]:
import json
import sys

import pandas as pd

from data_paths import DATA_DIR, METRICS_PATH, REPO_ROOT

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from agent.metrics import database as open_database
from agent.metrics import eligible_games

database = METRICS_PATH
with open_database(database, readonly=True) as db:
    runs = eligible_games(db)

## Results by model

`Wins` is based on how many games played by each model. `Request latency (s)` averages the median request time of model calls from each game.

`Divergences` is the number of model calls when a lower-scoring candidate is selected with a different action from the top backend proposal.

Compare `Minutes` with `Calls` and `Request latency` to see whether longer games involved more calls or slower requests.

In [ ]:
games = []
for run in runs:
    games.append({
        "Trace ID": run["id"][:8], "Saved at": pd.to_datetime(run["saved_at"], utc=True).tz_convert(
            "America/Los_Angeles").strftime("%Y-%m-%d %H:%M %Z"),
        "# decisions": run["decisions"], "Profile": run["profile"] or "Unknown",
        "Model": run["model_id"] or "Unknown", "Outcome": "Win" if run["result"] == "success" else "Loss",
        "Gold": run["gold"], "Calls": run["calls"], "Divergences": run["divergences"], "Scored": run["scored_calls"],
        "Minutes": run["wall_ms"] / 60000 if run["wall_ms"] is not None else None,
        "Request latency (s)": run["median_ms"] / 1000 if run["median_ms"] is not None else None,
        "trace": run["id"],
    })

def complete_mean(series):
    return series.mean() if series.notna().all() else None

def show(frame, formats=None):
    frame = frame.copy()
    if "Model" in frame and "Profile" in frame:
        frame["Model"] = [
            f"{profile}:{model}" if profile and model else model
            for profile, model in zip(frame["Profile"], frame["Model"])
        ]
    frame = frame.drop(columns=["Profile", "API"], errors="ignore")
    display(frame.style.hide(axis="index").format(
        {"Divergence rate": "{:.1%}", **(formats or {})}, precision=1, na_rep="Unknown"))


trend = pd.DataFrame(games)
if games:
    if trend["Trace ID"].duplicated().any():
        raise ValueError("Trace IDs share an eight-character prefix")
    grouped = trend.groupby(["Profile", "Model"], as_index=False)
    overview = grouped.agg(
        Wins=("Outcome", lambda outcomes: f"{outcomes.eq('Win').sum()}/{len(outcomes)}"),
        **{f"{col}/game": (col, complete_mean) for col in
           ("Gold", "Calls", "Minutes")},
        **{"Request latency (s)": ("Request latency (s)", complete_mean),
           "Divergences": ("Divergences", lambda values: values.sum(min_count=1)),
           "Scored": ("Scored", "sum"), "Games": ("Outcome", "size")},
    )
    overview["Divergence rate"] = overview["Divergences"] / overview.pop("Scored").replace(0, float("nan"))
    overview["Divergences"] = overview["Divergences"] / overview.pop("Games")
    overview = overview.rename(columns={"Divergences": "Divergences/game"})
    print("Table 1: Gameplay and timing by model")
    show(overview, {"Divergences/game": "{:.1f}"})
else:
    print("No eligible SQLite games.")

## Results by game

Newest run appears first. `Minutes` measures the whole game. Model `request latency` is the median for that game. Each game step counts as one `decision`, including those without a model `call`. [Token usage and cost](compare-cost.ipynb)

In [ ]:
trace_path = DATA_DIR / "agent-traces.json"
trace_ids = set(json.loads(trace_path.read_text())["runs"]) if trace_path.exists() else set()
if games:
    print("Table 2: Game results by save time")
    per_game = trend[trend["trace"].isin(trace_ids)].drop(columns="trace").copy()
    rate = per_game["Divergences"] / per_game.pop("Scored").replace(0, float("nan"))
    per_game.insert(per_game.columns.get_loc("Divergences") + 1, "Divergence rate", rate)
    per_game = per_game[[col for col in per_game if col not in ("Divergences", "Divergence rate")]
                        + ["Divergences", "Divergence rate"]]
    show(per_game)